# Encoder Checkpoint Evals

Runs the 9 encoder evals on each encoder checkpoint to compare quality across epochs.
Produces a DataFrame and optional plot for selecting the best checkpoint.

In [1]:
import sys
import json
import gc
import torch
import random
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from biojepa_v0_7 import BioJepa, BioJepaConfig
from training_v0_7 import create_model, maybe_compile
from config_v0_7 import VERSION
from evals.evals import EvalContext, run_encoder_evals, summarize_encoder_evals

## Config

In [2]:
SEED = 1337

def get_device():
    device = 'cpu'
    if torch.cuda.is_available():
        torch.cuda.manual_seed(SEED)
        device = 'cuda'
    print(f'using {device}')
    return torch.device(device)

torch.manual_seed(SEED)
random.seed(SEED)
torch.set_float32_matmul_precision('high')

device = get_device()

USE_COMPILE = torch.cuda.is_available()

data_root = Path('~/data/jepa/v0_7').expanduser()
ref_root = Path('~/data/jepa/reference_data').expanduser()
checkpoint_dir = data_root / 'checkpoint'

EVAL_SPLIT = 'val'
BATCH_SIZE = 64

using cpu


## Model Config

In [3]:
model_cfg = BioJepaConfig(
    num_genes=10000,
    n_layer= 6,
    heads= 4,
    embed_dim= 256,
    mlp_ratio=4.0,
    n_pre_layer=2,
    mask_ratio=0.766,
    gaussian_scale=2.38,
    film_linear_multiple=0.81,
    sim_coeff=50,
    std_coeff=25,
    cov_coeff=1,
    pert_latent_dim=128,
    pert_mode_dim=64,
)

## Discover Checkpoints

In [4]:
import re

epoch_pattern = re.compile(rf'biojepa_{VERSION}_encoder_epoch_(\d+)_step(\d+)\.pt')
final_pattern = re.compile(rf'biojepa_{VERSION}_encoder_final\.pt')

checkpoints = []
for p in sorted(checkpoint_dir.glob(f'biojepa_{VERSION}_encoder_*.pt')):
    m = epoch_pattern.match(p.name)
    if m:
        checkpoints.append({'path': p, 'epoch': int(m.group(1)), 'step': int(m.group(2)), 'label': f'epoch_{m.group(1)}'})
    elif final_pattern.match(p.name):
        checkpoints.append({'path': p, 'epoch': float('inf'), 'step': float('inf'), 'label': 'final'})

checkpoints.sort(key=lambda c: c['step'])
print(f'Found {len(checkpoints)} checkpoints:')
for c in checkpoints:
    print(f'  {c["label"]}: {c["path"].name}')

Found 3 checkpoints:
  epoch_26: biojepa_v0_7_encoder_epoch_26_step1651519.pt
  epoch_28: biojepa_v0_7_encoder_epoch_28_step1778559.pt
  epoch_30: biojepa_v0_7_encoder_epoch_30_step1905599.pt


## Run Evals on Each Checkpoint

In [5]:
eval_config = {
    'num_genes': model_cfg.num_genes, 'embed_dim': model_cfg.embed_dim,
    'n_layer': model_cfg.n_layer, 'heads': model_cfg.heads,
    'batch_size': BATCH_SIZE, 'verbose': False, 'seed': SEED,
    'eval_split': EVAL_SPLIT,
}

all_results = {}

for ckpt in checkpoints:
    label = ckpt['label']
    print(f'\n{"=" * 50}')
    print(f'{label}: {ckpt["path"].name}')

    model = create_model(model_cfg, device)
    model = maybe_compile(model, USE_COMPILE)

    with torch.serialization.safe_globals([BioJepaConfig]):
        checkpoint = torch.load(ckpt['path'], map_location=device)

    state_dict = checkpoint['model']
    if not USE_COMPILE and any('_orig_mod.' in k for k in state_dict):
        state_dict = {k.replace('_orig_mod.', ''): v for k, v in state_dict.items()}
    model.load_state_dict(state_dict)
    model.eval()

    eval_ctx = EvalContext(config=eval_config, data_root=data_root, checkpoint_root=data_root, ref_dir=ref_root)
    eval_ctx._biojepa = model

    raw_results = run_encoder_evals(eval_ctx)
    metrics = summarize_encoder_evals(raw_results)
    all_results[label] = metrics
    print(f'{label} metrics: {metrics}')

    eval_ctx._biojepa = None
    del eval_ctx, model, checkpoint, state_dict
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


epoch_26: biojepa_v0_7_encoder_epoch_26_step1651519.pt
found 66 shards for split val


KeyboardInterrupt: 

## Results

In [ ]:
df = pd.DataFrame(all_results).T
df.index.name = 'checkpoint'
df

In [ ]:
output_path = data_root / 'eval_results' / 'encoder_checkpoint_evals.json'
output_path.parent.mkdir(parents=True, exist_ok=True)
output_path.write_text(json.dumps(all_results, indent=2))
print(f'Saved to {output_path}')

## Plot Metrics Across Checkpoints

In [ ]:
plot_df = df.drop(index='final', errors='ignore')
plot_df.index = [int(x.replace('epoch_', '')) for x in plot_df.index]
plot_df = plot_df.sort_index()

if 'final' in df.index:
    plot_df.loc[plot_df.index.max() + 1] = df.loc['final']
    labels = [str(i) for i in plot_df.index[:-1]] + ['final']
else:
    labels = [str(i) for i in plot_df.index]

cols = [c for c in plot_df.columns if c != 'dead_dims']
n_cols = len(cols)
fig, axes = plt.subplots(1, n_cols, figsize=(4 * n_cols, 4))
if n_cols == 1:
    axes = [axes]

for ax, col in zip(axes, cols):
    vals = plot_df[col].dropna()
    ax.plot(range(len(vals)), vals.values, 'o-')
    ax.set_xticks(range(len(vals)))
    ax.set_xticklabels([labels[i] for i in range(len(vals))], rotation=45)
    ax.set_title(col)
    ax.set_xlabel('Epoch')

plt.tight_layout()
plt.show()